# 08 — Brain states in space and time (material for Figure 1)

The state space and the states as they unfold during a film. Pick the state set in the setup cell
and the cohort, film and state below.

| plot | what |
|---|---|
| 8.0 | latent space, one participant in colour over everyone in grey |
| 8.1 | latent space by film (task) |
| 8.2 | latent space by brain state |
| 8.3 | every viewer's state over one film, and the share of viewers in each state moment by moment |
| 8.4 | one state in detail: the regions most active in it over the film, and its profile against the other states |
| 8.5 | diagnostics: correlations against the volumes in a window, per cohort |
| 8.6 | diagnostics: the official space by cohort, before and after a per-person z-score |
| 8.7 | diagnostics: the per-TR activation space (window −1, as STIM) |
| 8.8 | diagnostics: UMAP refits (all windows, no z-score, z-score per person, NNDb only) |
| 8.9 | diagnostics: how much of each space person, task and cohort explain |
| 8.10 | diagnostics: one film's trajectory, people averaged moment by moment (STIM's view), 30-s and per-TR |
| 8.11 | diagnostics: UMAP on every training window, one panel per window size |
| 8.12 | diagnostics: Mapper shape graphs (STIM's figure) of the spaces above |
| 8.13 | Cam-CAN's space coloured by HADS anxiety and depression, every window size |


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import cohorts as cp

# ---- THE STATE SET: any atlas x window x clusterer stage 4b wrote (listed below)
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "HMM2_pca3_8", "ThresholdCluster_umap3_8", "MeanShift_umap3_3"

# ---- panel C: data balance. "time" = every cohort the same HOURS of scan per draw (whole scans,
# pooled transition counts): people and stimulus duration balanced together. None = cohort means.
BALANCE  = "time"
HOURS    = 10
VERSUS   = "mean"                  # C: each cohort vs the mean of the others; "pairs" = every pair
N_BOOT   = 200                     # draws for panel C, bootstrap over people for panel D
PVALUE   = "stars"                 # Figure 3C confidence column: None, "exact" or "stars"; marker hollow when P(tier) < .95
TEST     = "tier"                  # "tier": P(tier), share of bootstrap |d| in the observed tier (* ≥.90, ** ≥.95, *** ≥.99);
                                    # "difference" (paired t), "tost" (equivalence ±0.2 SD), "both", "all"
COMBINE  = "random"                # "mixed": one mixed model, task share of variance; or ω² per cohort combined: "random", "fixed", "equal"
GUIDES   = None                    # Figure 3 task x person: dashed lines at (.01, .06) with True, or a tuple; None = none
SAVE_DIR = None                    # e.g. "figures/14": PNG + SVG of every figure

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",
    distinct_minutes="min",   # distinct transitions rarefied to the shortest scan (None = as stored)
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",          # Rest, REST1..4 (HCP); every other task is movie
    alpha=0.10,                        # BH-FDR (A.2, films)
    fc_task=1.0, fc_dyn=0.25,          # A.2: |log2FC| for the graph and the dynamics
    edge_top=12,                       # A.1/A.2: strongest arrows drawn
    sesoi_d=0.2,
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; at {ATLAS}, {WINDOW_S:g} s:")
print(", ".join(sets[(sets.atlas == ATLAS) & (sets.window_s.astype(float) == WINDOW_S)]["states"]))
co = ct.Cohorts(cfg)
co.tasks()

In [ ]:
# ---- what the plots show
EMBEDDING   = "umap3"          # "umap3" | "pca3"
MAX_WINDOWS = 150              # windows per person drawn in the latent plots (at random)
WHO         = {"camcan": 1}    # 8.0: {cohort: n-th subject alphabetically, or a subject id}
COHORT      = "camcan"         # 8.3, 8.4: the cohort ...
TASK        = "Movie"          #   ... and the film (a task= value of that cohort)
STATE       = 7                # 8.4: the state shown in detail (0 .. K-1)
N_REGIONS   = 18               # 8.4: regions in the profile (half most, half least active in STATE)
N_TRACES    = 3                # 8.4: regions drawn over time
MAX_MINUTES = 10               # 8.3, 8.4: the first minutes of the film shown (None = all)
lat = ct.load_latents(co, EMBEDDING, MAX_WINDOWS)


#### Plot 8.0 — latent space, one participant


In [ ]:
# Plot 8.0
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# cp.fig_latents(co, lat, by="single", who=WHO, embedding=EMBEDDING, paths=True).show()

#### Plot 8.1 — latent space by film


In [ ]:
# Plot 8.1
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# cp.fig_latents(co, lat, by="task", embedding=EMBEDDING).show()

#### Plot 8.2 — latent space by brain state, training cohorts | projected cohorts


In [ ]:
# Plot 8.2 — two panels: the training cohorts together, the projected cohorts together
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# co_r, lat_r = ct.by_role(co, lat)
# cp.fig_latents(co_r, lat_r, by="cluster", embedding=EMBEDDING).show()

#### Plot 8.3 — the states over one film

Left: one row per viewer, one column per 30-s window (its start in film time, the same clock for every viewer); the strip on top is the most common state at each moment. Right: the share of viewers in each state, moment by moment. Vertical bands = the film drives everyone into the same state.

In [ ]:
# Plot 8.3
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# tl = ct.state_timeline(co, COHORT, TASK)
# if MAX_MINUTES:
#     tl = tl[tl["stimulus_start_s"] < 60 * MAX_MINUTES]
# print(f"{tl['sub'].nunique()} viewers, {tl['stimulus_start_s'].nunique()} windows")
# cp.fig_state_timeline(co, tl, f"{cp.cname(COHORT)} · {TASK} · every viewer's state", name=f"state_timeline_{COHORT}_{TASK}");

#### Plot 8.4 — one state in detail

Right: mean activation of each region (rows, clustered) in every state (columns), the regions most and least active in `STATE`; its column framed. Left: the `N_TRACES` regions most active in `STATE` over the film (mean over viewers, band = SD), on the share of viewers in `STATE` (grey). Activation as in the state maps (per-TR, band-passed 0.01–0.1 Hz, z-scored per run); the first read of a cohort's frames is cached.

In [ ]:
# Plot 8.4
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# F, M, tl = ct.state_activation(co, COHORT, TASK)      # M: the state profiles over the WHOLE film
# if MAX_MINUTES:                                        # only the time courses are cut
#     F = F[F["t"] < 60 * MAX_MINUTES]
#     tl = tl[tl["stimulus_start_s"] < 60 * MAX_MINUTES]
# cp.fig_state_detail(co, F, M, tl, STATE, n_regions=N_REGIONS, n_traces=N_TRACES,
#                     title=f"State S{STATE} · {cp.cname(COHORT)} · {TASK}");

## Latent-space diagnostics

Why the 30-s dFC space looks the way it does: grouped by person, unlike STIM's (Xian et al.), and with the projected cohorts landing elsewhere than the training ones. Plots 8.5 and 8.8 and the node-strength tables read what `slurm/latent_diagnostics.sbatch` wrote (nothing heavy runs here):

```
sbatch --array=0-5 slurm/latent_diagnostics.sbatch stats base all_rows no_scaler person_scaled nndb_only \
       -- --atlas harvardoxford --window-s 30 --states HMM2_umap3_8
```

| test | question | plot |
|---|---|---|
| A | do the correlations of a short window (Cam-CAN: ~12 volumes in 30 s) carry more sampling noise? | 8.5 |
| B | where do the projected cohorts land in the training space? | 8.6 |
| C | what does a STIM-like space (per-TR activation, window −1) look like? | 8.7 |
| E | does the UMAP fitted on every window bring back the clusters by film and person? | 8.8, 8.9 |
| F | normalisation **before** the reduction: none, per edge (official), per edge within each person | 8.8, 8.9 |
| — | normalisation **after** the reduction: each person's coordinates z-scored (STIM) | 8.6, 8.9 |
| G | fitted on NNDb alone, as the old version | 8.8, 8.9 |
| D | node strength instead of activation for the state maps | notebook 04, cell below 8.9 |

In [ ]:
# ---- diagnostics
VARIANTS     = ["base", "all_rows", "no_scaler", "person_scaled", "nndb_only"]   # the refits to read
TR_STATES    = None     # a state set at window -1 for 8.7 (ct.latent_options(co, window_s=-1) lists them); None = none
TR_MAX       = 300      # frames per person drawn from the per-TR space
DIAG_COLOURS = ["cohort", "task", "sub"]      # 8.7 / 8.8 / 8.11: one figure per colour; also "role", "kind", "state"
FILMS        = [(COHORT, TASK)]               # 8.10: (cohort, task) pairs, one trajectory each
DIAG_WINDOWS = [30, 60, 120, 300]             # 8.11: the window sizes with an all_rows refit
print(ct.latent_options(co, window_s=-1))

#### Plot 8.5 — correlations against the volumes in a window

In [ ]:
# Plot 8.5
ws, ps = ct.window_stats(co)
display(ps.groupby("cohort")[["n_tr", "mean_r", "edge_var"]].median().round(3)
        .join(ws.groupby("cohort")[["mean_abs_r", "sd_r"]].median().round(3)).rename(index=cp.cname))
cp.fig_window_stats(co, ws, ps, name="8_5_window_stats");

#### Plot 8.6 — the official space by cohort, before and after a per-person z-score

Left: the official 30-s space, coloured by cohort (training and projected together). Right: the same windows after each person's coordinates are z-scored, STIM's normalisation after the reduction; every person's offset is gone, so whatever grouping remains is how people move around their own centre.

In [ ]:
# Plot 8.6
import pandas as pd
off = ct.as_panel(co, lat, "official")
off_both = pd.concat([off, ct.zscore_within(off).assign(variant="official_z")], ignore_index=True)
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# for colour in ["cohort", "task"]:
#     cp.fig_embeddings(co, off_both, colour=colour, ncols=2, name=f"8_6_official_{colour}",
#                       title=f"8.6 · official {cp.window_name(co.cfg)} space, before | after a per-person z-score · colour = {colour}").show()

#### Plot 8.7 — the per-TR activation space (window −1)

One dot per frame (at most `TR_MAX` per person): the space STIM builds from activation, here from our own fit at window −1 (band-passed, z-scored per run). Right: the same after the per-person z-score.

In [ ]:
# Plot 8.7
lat_tr = ct.load_latents(co, EMBEDDING, TR_MAX, window_s=-1, states=TR_STATES or STATES, extra=("stimulus_start_s",))
tr = ct.as_panel(co, lat_tr, "per-TR")
tr_both = pd.concat([tr, ct.zscore_within(tr).assign(variant="per-TR, z-scored per person")], ignore_index=True)
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# for colour in DIAG_COLOURS:          # time: only within one film, see 8.10
#     cp.fig_embeddings(co, tr_both, colour=colour, ncols=2, name=f"8_7_per_tr_{colour}",
#                       title=f"8.7 · per-TR activation space (window −1) · colour = {colour}").show()

#### Plot 8.8 — UMAP refits

The same sampled windows (at most 60 per person) through each refit: `base` repeats the official recipe and is the one to compare with; the others change one thing each.

In [ ]:
# Plot 8.8
E = ct.diagnostic_embeddings(co, VARIANTS)
for v, m in E.attrs["meta"].items():
    print(f"{v:14s} fit on {m['fit_rows']:,} of {m['n_train_windows']:,} windows of {', '.join(m['train_cohorts'])}; "
          f"z-score before: {m['scaler']}")
# OFF: not under analysis now (keeps the notebook light); uncomment to redraw
# for colour in DIAG_COLOURS + ["state"]:
#     cp.fig_embeddings(co, E, colour=colour, name=f"8_8_refits_{colour}",
#                       title=f"8.8 · UMAP refits · colour = {colour}").show()

#### Plot 8.9 — how much of each space person, task and cohort explain

η² = between-group sum of squares / total, over the three coordinates (0 = the groups overlap completely, 1 = each group is a point). *person*: within each cohort; *task | person*: the tasks of one person told apart (people with one task add nothing); *rest | person*: rest vs movie within a person. After the per-person z-score, person and cohort are 0 by construction; what matters there is task. Overlapping windows inflate person and task alike, so compare spaces, not values against zero.

In [ ]:
# Plot 8.9
spaces = [off, ct.zscore_within(off).assign(variant="official_z"), tr,
          ct.zscore_within(tr).assign(variant="per-TR, z-scored per person")]
spaces += [E, *[ct.zscore_within(g).assign(variant=f"{v}_z") for v, g in E.groupby("variant", sort=False)]]
shares = ct.variance_shares(pd.concat(spaces, ignore_index=True), "variant")
shares.index = [cp.VARIANT_NAMES.get(i, i.replace("_z", ", z-scored per person after")) for i in shares.index]
shares.round(3)

#### Node strength instead of activation (test D)

`latent_diagnostics.sbatch stats` also wrote each state's mean edge vector over the training cohorts' windows. Give one to notebook 04 as `STATES` (the path below): an edge table is turned into node strength per parcel, each parcel's mean connectivity in that state, and correlated with the same reference maps. `_centred` is that minus the mean of every training window, so a parcel's value says how much its connectivity rises in the state, the analogue of the activation maps, which are z-scored.

In [ ]:
for f in sorted(ct.diagnostics_dir(co).glob(f"state_edges_{STATES}_*.csv")):
    print(f)

#### Plot 8.10 — one film's trajectory, people averaged moment by moment

STIM's figure is not a cloud of individual frames: each person's embedding is z-scored, then averaged over people at every moment of ONE film (everyone sees the same scene at the same stimulus time), and that mean trajectory is what is coloured by time. Left: the individual windows of the film (each person z-scored); right: the mean over people per moment, joined in time order. Top: 30-s windows; bottom: per-TR frames (window −1). The table: ε² of the moment within each film — how much the film's timeline places a window once each person is z-scored (adjusted for the number of moments; 0 = not at all).

In [ ]:
# Plot 8.10
import pandas as pd
spaces_t = {"30-s windows": ct.as_panel(co, ct.load_latents(co, EMBEDDING, 100_000, extra=("stimulus_start_s",)), "30 s"),
            "per-TR frames": ct.as_panel(co, ct.load_latents(co, EMBEDDING, 100_000, window_s=-1,
                                                             states=TR_STATES or STATES, extra=("stimulus_start_s",)), "per-TR")}
for c, t in FILMS:
    for name, L in spaces_t.items():
        traj = ct.film_trajectory(L, c, t)
        cp.fig_embeddings(co, traj, colour="time", ncols=2, size={"group mean": 4}, lines=("group mean",),
                          name=f"8_10_trajectory_{c}_{t}_{name.split()[0]}",
                          title=f"8.10 · {cp.cname(c)} · {t} · {name} · colour = time in the film").show()
moments = pd.concat({name: ct.moment_share(ct.zscore_within(L, by=("cohort", "sub", "task")))
                     for name, L in spaces_t.items()}, names=["space"]).reset_index(level=0)
moments.assign(cohort=moments["cohort"].map(cp.cname)).pivot_table(
    index=["cohort", "task"], columns="space", values="epsilon2").round(3)

#### Plot 8.11 — UMAP on every training window, by window size

The one refit with a different shape at 30 s, at every window size (each its own sbatch run: `sbatch --array=0-0 slurm/latent_diagnostics.sbatch all_rows -- --atlas harvardoxford --window-s 60 --states HMM2_umap3_8`, and 120, 300). The table: the η² of 8.9 for the official fit and the all-windows refit at each size.

In [ ]:
# Plot 8.11
Ew, rows = [], []
for w in DIAG_WINDOWS:
    try:
        Ew.append(ct.diagnostic_embeddings(co, ["all_rows"], window_s=w).assign(variant=f"{w:g} s"))
    except SystemExit as e:
        print(e)
    try:
        o = ct.as_panel(co, ct.load_latents(co, EMBEDDING, MAX_WINDOWS, window_s=w), f"{w:g} s, official")
        rows.append(o)
    except SystemExit as e:
        print(f"{w:g} s official: {e}")
if Ew:
    Ew = pd.concat(Ew, ignore_index=True)
    for colour in DIAG_COLOURS + ["state"]:
        cp.fig_embeddings(co, Ew, colour=colour, ncols=len(Ew["variant"].unique()), name=f"8_11_all_rows_{colour}",
                          titles={f"{w:g} s": f"{w:g} s · UMAP on every training window" for w in DIAG_WINDOWS},
                          title=f"8.11 · UMAP on every training window, by window size · colour = {colour}").show()
    rows.append(Ew.assign(variant=Ew["variant"] + ", every window"))
ct.variance_shares(pd.concat(rows, ignore_index=True), "variant").round(3)

#### Plot 8.12 — Mapper shape graphs

STIM's figure: the 3-D embedding cut into `MAPPER_BINS`³ cubes overlapping by `MAPPER_OVERLAP`, the windows in each cube clustered (DBSCAN), each cluster a node (area = windows in it), two nodes joined when they share windows. A summary of the shape of the same UMAP, not a new space: it hides density and single points and keeps branches and loops. Here the clustering runs on the 3 coordinates (STIM clusters on the original 271 regions). Top: the official 30-s and the per-TR spaces (at most 30,000 windows each), by state and by cohort. Bottom: one film's individual windows and group trajectory (8.10, `FILMS[0]`), by time in the film. Clean-up for drawing: nodes with fewer than `MAPPER_MIN_NODE` windows and edges whose shared windows are less than `MAPPER_MIN_JACCARD` of the two nodes' union are dropped; without it a dense cloud draws as the lattice of the cubes.


In [ ]:
# Plot 8.12
MAPPER_BINS, MAPPER_OVERLAP = 8, .5
MAPPER_MIN_NODE = 20       # drop nodes with fewer windows
MAPPER_MIN_JACCARD = .15   # drop edges whose shared windows are < 15% of the two nodes together
G = {}
for title, L in [("official 30-s space", off), ("per-TR space", tr)]:
    rows, nodes, edges = ct.mapper_graph(L, MAPPER_BINS, MAPPER_OVERLAP, min_node=MAPPER_MIN_NODE, min_jaccard=MAPPER_MIN_JACCARD)
    G[title] = (rows, nodes, edges, ct.graph_layout(nodes, edges, rows))
for colour in ["state", "cohort"]:
    cp.fig_mapper(co, G, colour, name=f"8_12_mapper_{colour}")
c, t = FILMS[0]
GT = {}
for name, L in spaces_t.items():
    rows, nodes, edges = ct.mapper_graph(ct.film_trajectory(L, c, t), MAPPER_BINS, MAPPER_OVERLAP, min_samples=2,
                                        min_node=3, min_jaccard=MAPPER_MIN_JACCARD)
    GT[f"{cp.cname(c)} · {t} · {name}"] = (rows, nodes, edges, ct.graph_layout(nodes, edges, rows))
cp.fig_mapper(co, GT, "time", name=f"8_12_mapper_trajectory_{c}_{t}");

#### Plot 8.13 — anxiety and depression in the space, by window size

Cam-CAN only (the one cohort with HADS), every window size of the official fit and the per-TR space. First the windows (at most `MAX_WINDOWS` per person), then one dot per person (their mean position), coloured by the raw HADS score (0–21). The table: R² of each score on a person's mean coordinates, in sample and 5-fold cross-validated (≤ 0 = the position says nothing about the score).

In [ ]:
# Plot 8.13
HADS_WINDOWS = [30, 60, 120, 300, -1]
scores = ct.hads_scores(raw=True)
parts = []
for w in HADS_WINDOWS:
    try:
        L = ct.as_panel(co, ct.load_latents(co, EMBEDDING, MAX_WINDOWS, window_s=w, states=TR_STATES if w == -1 else None),
                        "per-TR" if w == -1 else f"{w:g} s")
        parts.append(L[L["cohort"] == "camcan"])
    except SystemExit as e:
        print(f"{w:g}: {e}")
H = pd.concat(parts, ignore_index=True).merge(scores, on="sub", how="left")
print(f"{H['sub'].nunique()} Cam-CAN people, {H.drop_duplicates('sub')['anxiety'].notna().sum()} with an anxiety score")
P = H.groupby(["variant", "cohort", "sub"], sort=False, as_index=False)[["x", "y", "z", "anxiety", "depression"]].mean() \
     .assign(task="mean")
for s in ["anxiety", "depression"]:
    cp.fig_embeddings(co, H, colour=s, ncols=len(parts), height=420, name=f"8_13_{s}_windows",
                      title=f"8.13 · Cam-CAN windows · colour = HADS {s}").show()
    cp.fig_embeddings(co, P, colour=s, ncols=len(parts), height=420, size=4, name=f"8_13_{s}_people",
                      title=f"8.13 · Cam-CAN, one dot per person (mean position) · colour = HADS {s}").show()
ct.symptom_fit(H, scores).round(3)